# 06. Vision-Language & LLM Reasoning (Gemma 4)

## 1. Overview
- **Task**: Multi-modal Attribute Tagging & AI Investigator Reasoning
- **Tool**: Google Gemma 4 (31B parameter Vision-Language Model)
- **License**: Google Gemma Terms of Use (Permissive, free for commercial use, no recurring license fees)
- **Runtime**: Served via `vLLM` on AMD ROCm (MI300X) with an OpenAI-compatible API (`/v1/chat/completions`).

### Two Primary Architectural Roles
1. **Explainable Semantic Tagging**: Generates structured human-readable descriptors (e.g. clothing color, garment type, accessories) for person and vehicle crops. This gives human operators verifiable evidence rather than relying only on opaque embedding distances.
2. **AI Investigator Chat**: Answers operator inquiries in natural Thai or English ("Who entered the warehouse between 14:00 and 15:00?") by calling "fat tools" over ClickHouse surveillance event logs, enforced by deterministic grounding gates to eliminate hallucinations.

In [ ]:
# Import dependencies
import base64
import io
import json
import os
import cv2
import numpy as np
from PIL import Image

## 2. Image Encoding & Prompt Formatting

In [ ]:
def encode_image_base64(bgr_image: np.ndarray) -> str:
    """Converts OpenCV BGR image array into base64 JPEG string."""
    rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    buffer = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=85)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

PERSON_PROMPT_TEMPLATE = (
    "Describe the person in this CCTV crop for surveillance tracking.\n"
    "Respond strictly with a JSON object in this format:\n"
    "{\n"
    "  \"upper_color\": \"<color>\",\n"
    "  \"lower_color\": \"<color>\",\n"
    "  \"garment\": \"<shirt|t-shirt|jacket|hoodie|uniform>\",\n"
    "  \"accessories\": [\"<backpack|hat|glasses|none>\"],\n"
    "  \"summary\": \"<short description>\"\n"
    "}\n"
    "Do not invent details you cannot clearly see."
)

## 3. VLM Client Wrapper (vLLM / OpenAI Compatible)

In [ ]:
class GemmaVLMClient:
    """Client for interacting with Gemma 4 Vision-Language model."""

    def __init__(self, endpoint_url: str = 'http://localhost:8000/v1', model_name: str = 'gemma-4-31b-it'):
        self.endpoint_url = endpoint_url
        self.model_name = model_name
        self.client = None
        try:
            from openai import OpenAI
            self.client = OpenAI(base_url=endpoint_url, api_key='none')
        except Exception:
            print('OpenAI client library not available. Using mock generation mode.')

    def describe_person(self, bgr_crop: np.ndarray) -> dict:
        if self.client is None:
            # Return realistic mock JSON for offline testing
            return {
                'upper_color': 'black',
                'lower_color': 'blue',
                'garment': 'hoodie',
                'accessories': ['backpack'],
                'summary': 'Person in black hoodie and blue jeans carrying a backpack'
            }

        b64 = encode_image_base64(bgr_crop)
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        'role': 'user',
                        'content': [
                            {'type': 'text', 'text': PERSON_PROMPT_TEMPLATE},
                            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}}
                        ]
                    }
                ],
                temperature=0.1,
                max_tokens=256
            )
            content = response.choices[0].message.content
            # Extract JSON block
            clean_content = content.replace('```json', '').replace('```', '').strip()
            return json.loads(clean_content)
        except Exception as e:
            print(f'VLM query failed: {e}. Falling back to default.')
            return {'summary': 'Unable to parse visual attributes'}

## 4. Grounding Gate: Preventing Hallucination in the Investigator

The system mandates that any Global ID, timestamp, or camera name mentioned by the LLM must match a deterministic record in ClickHouse.

In [ ]:
class GroundingGate:
    """Validates LLM statements against actual ground truth records."""

    def __init__(self, known_gids: set, known_cameras: set):
        self.known_gids = known_gids
        self.known_cameras = known_cameras

    def verify_answer(self, mentioned_gid: str, mentioned_cam: str) -> bool:
        if mentioned_gid not in self.known_gids:
            print(f"[Grounding Alert] Hallucinated Global ID rejected: {mentioned_gid}")
            return False
        if mentioned_cam not in self.known_cameras:
            print(f"[Grounding Alert] Unknown Camera rejected: {mentioned_cam}")
            return False
        return True

# Demonstration
ground_gate = GroundingGate(known_gids={'G1001', 'G1002'}, known_cameras={'cam_lobby', 'cam_corridor_1'})
vlm = GemmaVLMClient()

# Create dummy crop
dummy_person = np.zeros((200, 100, 3), dtype=np.uint8)
attributes = vlm.describe_person(dummy_person)
print('Extracted Visual Semantic Attributes:')
print(json.dumps(attributes, indent=2))

# Verification test
is_valid = ground_gate.verify_answer('G1001', 'cam_lobby')
print(f"Grounding verification for G1001 at cam_lobby: {'PASSED' if is_valid else 'FAILED'}")

is_invalid = ground_gate.verify_answer('G9999', 'cam_basement')
print(f"Grounding verification for hallucinated record: {'PASSED' if is_invalid else 'REJECTED'}")